In [229]:
import pandas as pd
import numpy as np
import re
import glob

In [230]:
df= pd.read_csv("../raw_data/amazon_india_2018.csv")

In [231]:
df.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
0,TXN_2018_00000001,2018-01-28,CUST_2015_00011381,PROD_000367,Samsung Galaxy S9+ 128GB White,Electronics,Smartphones,Samsung,123828.89,0.0,...,False,NaN,4.5,Delivered,1,2018,1,0.21,False,3.9
1,TXN_2018_00000002,2018-01-09,CUST_2018_00004073,PROD_001683,Apple Mi Pad 8GB RAM Silver,Electronics,Tablets,Apple,61478.19,0.0,...,False,NaN,4.0,Delivered,1,2018,1,0.40,True,4.5
2,TXN_2018_00000003,2018-01-06,CUST_2018_00024729,PROD_001688,Apple Slate 8GB RAM Black,Electronics,Tablets,Apple,38624.59,0.0,...,False,NaN,4.5,Delivered,1,2018,1,0.41,True,3.2
3,TXN_2018_00000004,2018-01-20,CUST_2018_00015834,PROD_000236,Apple iPhone X 64GB Blue,Electronics,Smartphones,Apple,201630.77,47.2,...,True,Republic Day Sale,5.0,Delivered,1,2018,1,0.25,True,4.0
4,TXN_2018_00000005,2018-01-07,CUST_2016_00001228,PROD_000163,OnePlus OnePlus 3T 16GB Black,Electronics,Smartphones,OnePlus,53961.7,0.0,...,False,NaN,NaN,Delivered,1,2018,1,0.16,True,4.7


In [232]:
df["delivery_charges"].isna().sum(), len(df)

(np.int64(7953), 99495)

In [233]:
df["delivery_charges"].describe()


count    91542.0
mean         0.0
std          0.0
min          0.0
25%          0.0
50%          0.0
75%          0.0
max          0.0
Name: delivery_charges, dtype: float64

In [234]:
df.drop(columns=["delivery_charges"], inplace=True)

In [235]:
df.shape

(99495, 33)

In [236]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99495 entries, 0 to 99494
Data columns (total 33 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   transaction_id          99495 non-null  object 
 1   order_date              99495 non-null  object 
 2   customer_id             99495 non-null  object 
 3   product_id              99495 non-null  object 
 4   product_name            99495 non-null  object 
 5   category                99495 non-null  object 
 6   subcategory             99495 non-null  object 
 7   brand                   99495 non-null  object 
 8   original_price_inr      99495 non-null  object 
 9   discount_percent        99495 non-null  float64
 10  discounted_price_inr    99495 non-null  float64
 11  quantity                99495 non-null  int64  
 12  subtotal_inr            99495 non-null  float64
 13  final_amount_inr        99495 non-null  float64
 14  customer_city           99495 non-null

In [237]:
df.columns

Index(['transaction_id', 'order_date', 'customer_id', 'product_id',
       'product_name', 'category', 'subcategory', 'brand',
       'original_price_inr', 'discount_percent', 'discounted_price_inr',
       'quantity', 'subtotal_inr', 'final_amount_inr', 'customer_city',
       'customer_state', 'customer_tier', 'customer_spending_tier',
       'customer_age_group', 'payment_method', 'delivery_days',
       'delivery_type', 'is_prime_member', 'is_festival_sale', 'festival_name',
       'customer_rating', 'return_status', 'order_month', 'order_year',
       'order_quarter', 'product_weight_kg', 'is_prime_eligible',
       'product_rating'],
      dtype='object')

Question 1
Your dataset contains order_date in multiple formats: 'DD/MM/YYYY', 'DD-MM-YY', 'YYYY-MM-DD', and some invalid entries like '32/13/2020'. Clean and standardize all dates to 'YYYY-MM-DD' format, handling invalid dates appropriately.


In [238]:
df["order_date"].head(20)

0     2018-01-28
1     2018-01-09
2     2018-01-06
3     2018-01-20
4     2018-01-07
5     2018-01-30
6     2018-01-27
7     2018-01-23
8     2018-01-08
9     2018-01-11
10    26-01-2018
11    2018-01-26
12    2018-01-12
13    2018-01-02
14    2018-01-04
15    12/01/2018
16    2018-01-06
17    2018-01-26
18    2018-01-25
19    2018-01-16
Name: order_date, dtype: object

In [239]:
df["order_date"]=(
    df["order_date"]
    .str.replace("/" , "-", regex=False)
    .str.replace(" ", "", regex=False)
    
    )

parts= df["order_date"].str.split("-", expand=True)
year_last= parts[2].str.len()==4
df.loc[year_last,"order_date"]= (parts[2]+"-"+parts[0]+"-"+parts[1])

parts= df["order_date"]. str. split("-", expand=True)
mask= parts[1].astype(int) > 12
df.loc[mask, "order_date"]= (parts[0]+"-"+parts[2]+"-"+parts[1])



In [240]:
df["order_date"]= pd.to_datetime(df["order_date"], errors="coerce")

In [241]:
df["order_date"].isna().sum()

np.int64(0)

Question 2
The original_price_inr column contains mixed data types: numeric values, text with '₹' symbols, comma separators ('₹1,25,000'), and some entries like 'Price on Request'. Clean this column to contain only numeric values in Indian Rupees. 


In [242]:
df["original_price_inr"] = df["original_price_inr"].astype(str)

df["original_price_inr"] = df["original_price_inr"].str.replace("₹", "", regex=False)

df["original_price_inr"] = df["original_price_inr"].str.replace(",", "", regex=False)

df["original_price_inr"] = pd.to_numeric(df["original_price_inr"], errors="coerce")

In [243]:
df["original_price_inr"].unique()[:20]

array([123828.89,  61478.19,  38624.59, 201630.77,  53961.7 ,  26265.87,
        48227.04,  22399.73,  48472.48, 234658.9 ,  31018.55,  59337.94,
       142248.89,  29555.99,  32633.32, 174786.05, 246998.55,  49034.97,
        55879.27,  29251.66])

In [244]:
df["original_price_inr"].dtypes

dtype('float64')

Question 3
Customer ratings appear in various formats: '5.0', '4 stars', '3/5', '2.5/5.0', and some missing values. Standardize all ratings to numeric scale 1.0-5.0, handling inconsistent formats and missing values strategically.


In [245]:
df["customer_rating"]=df["customer_rating"].astype(str)
df["customer_rating"]= df["customer_rating"].str.replace("stars","",regex=False)
df["customer_rating"]= df["customer_rating"].str.split("/").str[0]
df["customer_rating"]= pd.to_numeric(df["customer_rating"],errors="coerce")


In [246]:
df["customer_rating"].describe()

count    69261.000000
mean         4.315538
std          0.572839
min          3.000000
25%          4.000000
50%          4.500000
75%          5.000000
max          5.000000
Name: customer_rating, dtype: float64

In [247]:
df["customer_rating"].value_counts().head(10)

customer_rating
4.5    22841
5.0    18038
4.0    17320
3.5     6916
3.0     4146
Name: count, dtype: int64

In [248]:
df["customer_rating"].isna().sum()

np.int64(30234)

In [249]:
df["customer_rating"].value_counts(dropna=False)

customer_rating
NaN    30234
4.5    22841
5.0    18038
4.0    17320
3.5     6916
3.0     4146
Name: count, dtype: int64

Question 4
The customer_city column has inconsistent naming: 'Bangalore/Bengaluru', 'Mumbai/Bombay', 'Delhi/New Delhi', along with spelling errors and case variations. Standardize all city names and handle geographical variations.


In [250]:
df["customer_city"]= df["customer_city"].str.strip().str.lower()

In [251]:
df["customer_city"].unique()

array(['delhi', 'mumbai', 'kochi', 'indore', 'bangalore', 'kanpur',
       'hyderabad', 'ludhiana', 'chandigarh', 'visakhapatnam', 'pune',
       'surat', 'lucknow', 'saharanpur', 'coimbatore', 'jaipur',
       'ahmedabad', 'kolkata', 'chennai', 'nagpur', 'patna',
       'bhubaneswar', 'vadodara', 'aligarh', 'bombay', 'moradabad',
       'meerut', 'varanasi', 'gorakhpur', 'madras', 'bengaluru',
       'bareilly', 'delhi ncr', 'allahabad', 'banglore', 'mumba',
       'chenai', 'calcutta', 'bengalore', 'new delhi'], dtype=object)

In [252]:
city_map = {
    "bombay": "mumbai",
    "mumba": "mumbai",

    "madras": "chennai",
    "chenai": "chennai",

    "calcutta": "kolkata",

    "bengaluru": "bangalore",
    "bengalore": "bangalore",
    "banglore": "bangalore",

    "new delhi": "delhi",
    "delhi ncr": "delhi"
}

In [253]:
df["customer_city"] = df["customer_city"].replace(city_map)

In [254]:
df["customer_city"] = df["customer_city"].str.title()

In [255]:
df["customer_city"].value_counts().head()


customer_city
Mumbai       14111
Delhi        12621
Bangalore    10489
Chennai       8679
Kolkata       6848
Name: count, dtype: int64

In [256]:
df["customer_city"].unique()


array(['Delhi', 'Mumbai', 'Kochi', 'Indore', 'Bangalore', 'Kanpur',
       'Hyderabad', 'Ludhiana', 'Chandigarh', 'Visakhapatnam', 'Pune',
       'Surat', 'Lucknow', 'Saharanpur', 'Coimbatore', 'Jaipur',
       'Ahmedabad', 'Kolkata', 'Chennai', 'Nagpur', 'Patna',
       'Bhubaneswar', 'Vadodara', 'Aligarh', 'Moradabad', 'Meerut',
       'Varanasi', 'Gorakhpur', 'Bareilly', 'Allahabad'], dtype=object)

Question 5
Boolean columns (is_prime_member, is_prime_eligible, is_festival_sale) contain mixed values: True/False, Yes/No, 1/0, Y/N, and some missing entries. Convert all boolean columns to consistent True/False format.


In [257]:
bool_candidates=[]
bool_values={"true", "false","y","n","yes", "no","0","1"}

for col in df.columns:
    vals= set(df[col].astype(str). str.lower().dropna().unique())
    if vals & bool_values:
        bool_candidates.append(col)
bool_candidates


['quantity',
 'delivery_days',
 'is_prime_member',
 'is_festival_sale',
 'order_month',
 'order_quarter',
 'is_prime_eligible']

In [258]:
boolean_cols = ["is_prime_member", "is_prime_eligible", "is_festival_sale"]

bool_map = {
    "true": True,
    "false": False,
    "yes": True,
    "no": False,
    "y": True,
    "n": False,
    "1": True,
    "0": False
}

for col in boolean_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
        .str.lower()
        .map(bool_map)
    )

In [259]:
df[boolean_cols].value_counts(dropna=False)

is_prime_member  is_prime_eligible  is_festival_sale
False            True               False               47232
                                    True                22149
                 False              False                9764
True             True               False                8937
False            False              True                 4431
True             True               True                 4129
                 False              False                1985
                                    True                  868
Name: count, dtype: int64

Question 6
Product categories have variations: 'Electronics/Electronic/ELECTRONICS/Electronics & Accessories'. Standardize category names across the dataset and ensure consistent naming conventions.


In [260]:
df["category"].value_counts().head(20)

category
Electronics                  99415
Electronic                      28
Electronics & Accessories       23
Electronicss                    15
ELECTRONICS                     14
Name: count, dtype: int64

In [261]:
df["category"] = df["category"].str.strip().str.lower()


In [262]:
category_map = {
    "electronic": "electronics",
    "electronics": "electronics",
    "electronics & accessories": "electronics",
    "electronicss": "electronics"
}

In [263]:
df["category"] = df["category"].replace(category_map)
df["category"] = df["category"].str.title()

In [264]:
df["category"].value_counts()

category
Electronics    99495
Name: count, dtype: int64

Question 7
The delivery_days column contains negative values, text entries like 'Same Day', '1-2 days', and some unrealistic values like 50 days. Clean this column to contain only valid numeric delivery days.


In [265]:
df["delivery_days"].unique()

array(['3', '4', '7', '2', '1', '6', '5', '-1', '1-2 days', 'Express',
       '15', 'Same Day', '0'], dtype=object)

In [266]:
df["delivery_days"] = df["delivery_days"].astype(str).str.strip().str.lower()

In [267]:
df["delivery_days"] = df["delivery_days"].replace({
    "same day": "0",
    "express": "1"
})


In [268]:
df["delivery_days"] = df["delivery_days"].str.extract(r"(-?\d+)")

In [269]:
df["delivery_days"] = pd.to_numeric(df["delivery_days"], errors="coerce")

In [270]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = None

In [271]:
df["delivery_days"].unique()

array([ 3.,  4.,  7.,  2.,  1.,  6.,  5., nan, 15.,  0.])

In [272]:
df["delivery_days"].isnull().sum()

np.int64(601)

In [273]:
df.loc[df["delivery_days"] < 0, "delivery_days"] = np.nan

df["delivery_days"] = df["delivery_days"].fillna(df["delivery_days"].median())

In [274]:
df["delivery_days"].describe()

count    99495.000000
mean         3.905895
std          1.598470
min          0.000000
25%          3.000000
50%          4.000000
75%          5.000000
max         15.000000
Name: delivery_days, dtype: float64

In [275]:
df["delivery_days"].isna().sum()

np.int64(0)

Question 8
Identify and handle duplicate transactions where the same customer, product, date, and amount appear multiple times. Some duplicates are genuine (bulk orders) while others are data errors. Develop a strategy to distinguish and handle both cases.


In [276]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [277]:
duplicate_mask = df.duplicated(
    subset=["customer_id", "product_id", "order_date", "original_price_inr"],
    keep=False
)

duplicates = df[duplicate_mask]

In [278]:
duplicates.shape

(980, 33)

In [279]:
duplicates.head()

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
14,TXN_2018_00000015,2018-01-04,CUST_2018_00018940,PROD_000402,Xiaomi Poco F1 128GB Blue,Electronics,Smartphones,Xiaomi,32633.32,0.00,...,False,NaN,4.5,Delivered,1,2018,1,0.19,False,3.4
108,TXN_2018_00000109,2018-01-29,CUST_2018_00025678,PROD_000072,Xiaomi Redmi 2 64GB Black,Electronics,Smartphones,Xiaomi,23246.13,10.67,...,False,NaN,4.5,Delivered,1,2018,1,0.19,True,3.3
123,TXN_2018_00000124,2018-01-06,CUST_2018_00013289,PROD_000070,Xiaomi Redmi 2 16GB Black,Electronics,Smartphones,Xiaomi,32791.35,6.84,...,False,NaN,NaN,Delivered,1,2018,1,0.20,True,4.2
323,TXN_2018_00000324,2018-01-24,CUST_2016_00013330,PROD_000327,Oppo A71 64GB Blue,Electronics,Smartphones,Oppo,34769.83,39.33,...,True,Republic Day Sale,NaN,Delivered,1,2018,1,0.24,True,3.5
498,TXN_2018_00000499,2018-01-10,CUST_2017_00012045,PROD_001692,Samsung Galaxy Tab 8GB RAM Silver,Electronics,Tablets,Samsung,115081.13,14.57,...,False,NaN,5.0,Delivered,1,2018,1,0.43,False,4.8


In [280]:
df.duplicated().sum()

np.int64(0)

In [281]:
df = df.drop_duplicates()

In [282]:
duplicates.sort_values(
    ["customer_id","product_id","order_date"]
).head(10)

,transaction_id,order_date,customer_id,product_id,product_name,category,subcategory,brand,original_price_inr,discount_percent,...,is_festival_sale,festival_name,customer_rating,return_status,order_month,order_year,order_quarter,product_weight_kg,is_prime_eligible,product_rating
38888,TXN_2018_00038889,2018-06-02,CUST_2015_00000237,PROD_000063,OnePlus OnePlus X 32GB Blue,Electronics,Smartphones,OnePlus,83692.32,0.0,...,False,NaN,4.5,Delivered,6,2018,2,0.18,True,3.8
99243,TXN_2018_00038889_DUP,2018-06-02,CUST_2015_00000237,PROD_000063,OnePlus OnePlus X 32GB Blue,Electronics,Smartphones,OnePlus,83692.32,0.0,...,False,NaN,4.5,Delivered,6,2018,2,0.18,True,3.8
58252,TXN_2018_00058253,2018-08-19,CUST_2015_00001621,PROD_000249,Samsung Galaxy S8+ 32GB Blue,Electronics,Smartphones,Samsung,103835.79,0.0,...,False,NaN,NaN,Delivered,8,2018,3,0.15,False,3.2
99160,TXN_2018_00058253_DUP,2018-08-19,CUST_2015_00001621,PROD_000249,Samsung Galaxy S8+ 32GB Blue,Electronics,Smartphones,Samsung,103835.79,0.0,...,False,NaN,NaN,Delivered,8,2018,3,0.15,False,3.2
24097,TXN_2018_00024098,2018-04-22,CUST_2015_00002076,PROD_000122,Apple iPhone SE 16GB Black,Electronics,Smartphones,Apple,116766.99,0.0,...,False,NaN,5.0,Delivered,4,2018,2,0.19,True,3.6
99217,TXN_2018_00024098_DUP,2018-04-22,CUST_2015_00002076,PROD_000122,Apple iPhone SE 16GB Black,Electronics,Smartphones,Apple,116766.99,0.0,...,False,NaN,5.0,Delivered,4,2018,2,0.19,True,3.6
96354,TXN_2018_00096355,2018-12-05,CUST_2015_00002677,PROD_000411,Xiaomi Redmi 5A 128GB Blue,Electronics,Smartphones,Xiaomi,30156.73,0.0,...,False,NaN,5.0,Delivered,12,2018,4,0.23,False,4.5
99420,TXN_2018_00096355_DUP,2018-12-05,CUST_2015_00002677,PROD_000411,Xiaomi Redmi 5A 128GB Blue,Electronics,Smartphones,Xiaomi,30156.73,0.0,...,False,NaN,5.0,Delivered,12,2018,4,0.23,False,4.5
65384,TXN_2018_00065385,2018-09-16,CUST_2015_00002755,PROD_000259,Samsung Galaxy J7 Max 16GB Blue,Electronics,Smartphones,Samsung,37837.48,22.2,...,False,NaN,5.0,Delivered,9,2018,3,0.16,True,4.5
99141,TXN_2018_00065385_DUP,2018-09-16,CUST_2015_00002755,PROD_000259,Samsung Galaxy J7 Max 16GB Blue,Electronics,Smartphones,Samsung,37837.48,22.2,...,False,NaN,5.0,Delivered,9,2018,3,0.16,True,4.5


In [283]:
duplicates.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().sort_values(ascending=False).head(10)

customer_id         product_id   order_date  original_price_inr
CUST_2015_00000237  PROD_000063  2018-06-02  83692.32              2
CUST_2015_00001621  PROD_000249  2018-08-19  103835.79             2
CUST_2015_00002076  PROD_000122  2018-04-22  116766.99             2
CUST_2015_00002677  PROD_000411  2018-12-05  30156.73              2
CUST_2015_00002755  PROD_000259  2018-09-16  37837.48              2
CUST_2015_00002972  PROD_000036  2018-01-08  104592.57             2
CUST_2015_00003595  PROD_001975  2018-12-15  35455.60              2
CUST_2015_00003619  PROD_000267  2018-09-24  68144.26              2
CUST_2015_00003983  PROD_000287  2018-09-26  30661.00              2
CUST_2015_00004142  PROD_000303  2018-03-18  48032.92              2
dtype: int64

In [284]:
dup_groups = df.groupby(
    ["customer_id","product_id","order_date","original_price_inr"]
).size().reset_index(name="count")

dup_groups = dup_groups[dup_groups["count"] > 1]

In [285]:
dup_rows = df.merge(
    dup_groups,
    on=["customer_id","product_id","order_date","original_price_inr"],
    how="inner"
)

In [286]:
dup_rows[["customer_id","product_id","quantity","count"]].head()

,customer_id,product_id,quantity,count
0,CUST_2018_00018940,PROD_000402,1,2
1,CUST_2018_00025678,PROD_000072,3,2
2,CUST_2018_00013289,PROD_000070,1,2
3,CUST_2016_00013330,PROD_000327,1,2
4,CUST_2017_00012045,PROD_001692,3,2


In [287]:
df_clean = df.drop_duplicates(
    subset=["customer_id","product_id","order_date","original_price_inr"],
    keep="first"
)

In [288]:
df_clean.duplicated(
    subset=["customer_id","product_id","order_date","original_price_inr"]
).sum()

np.int64(0)

In [289]:
df["transaction_id"].duplicated().sum()

np.int64(0)

In [290]:
df = df.drop_duplicates(subset="transaction_id", keep="first")

Question 9
The dataset contains outlier prices where some products show prices 100x higher than expected due to data entry errors (decimal point issues). Identify and correct these outliers using statistical methods and domain knowledge.


In [291]:
# ── FIX 1: Negative prices ──────────────────────────
neg_mask = df["original_price_inr"] < 0
df.loc[neg_mask, "original_price_inr"] = df.loc[neg_mask, "original_price_inr"].abs()
print(f"Negative prices fixed: {neg_mask.sum()}")

# ── FIX 2: Outliers (100x decimal error) ────────────
subcategory_caps = {
    "Smart Watch":        50000,     # Amazfit, Fire-Boltt premium ~₹48k
    "Tablets":            80000,     # Lenovo Tab M10 ~₹79k
    "Smartphones":        150000,    # Samsung S9+, Note 9 flagship
    "Laptops":            200000,    # ASUS, Acer gaming laptops
    "TV & Entertainment": 260000,    # LG OLED, Samsung Smart TV
    "Audio":              40000,     # unchanged ✅
}

outlier_mask = df.apply(
    lambda row: row["original_price_inr"] > subcategory_caps.get(row["subcategory"], 999999),
    axis=1
)
df.loc[outlier_mask, "original_price_inr"] = (
    df.loc[outlier_mask, "original_price_inr"] / 100
).round(2)
print(f"Outliers fixed: {outlier_mask.sum()}")

# ── FIX 3: Recalculate ───────────────────────────────
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]

# ── VERIFY ───────────────────────────────────────────
print(df.groupby("subcategory", observed=True)["original_price_inr"]
      .describe()[["min","max","mean","50%"]].round(2))
print(f"\nNaN in final_amount_inr:   {df['final_amount_inr'].isna().sum()}")
print(f"Negative prices remaining: {(df['original_price_inr'] < 0).sum()}")


Negative prices fixed: 251
Outliers fixed: 11514
                        min        max       mean        50%
subcategory                                                 
Audio               2029.03   30102.04   19622.14   22737.46
Laptops             2049.36  223657.81   88910.24   86451.97
Smart Watch          517.71   59337.94   20543.84   24567.02
Smartphones         1500.59  242371.11   53156.18   40358.80
TV & Entertainment  3029.61  256740.30  118550.33  134785.84
Tablets              947.87  113586.17   39995.27   38624.59

NaN in final_amount_inr:   2982
Negative prices remaining: 0


In [292]:
# Check if any legitimate products were over-corrected in cleaned files
for sub, cap in subcategory_caps.items():
    over = df_clean[(df_clean["subcategory"] == sub) & 
                    (df_clean["original_price_inr"] > cap)]
    if len(over) > 0:
        print(f"\n{sub} (cap ₹{cap:,}): {len(over)} rows over")
        print(over[["product_name", "original_price_inr"]].drop_duplicates().head(5))
    else:
        print(f"\n{sub}: ✅ All within cap")


Smart Watch (cap ₹50,000): 1932 rows over
                    product_name  original_price_inr
11                  Fitbit Watch            59337.94
18   Garmin Sports Watch Premium            55879.27
169         Xiaomi Watch Premium            57458.14
267          Garmin Watch Deluxe            56103.32
407   Apple Fitness Band Premium            51770.80

Tablets (cap ₹80,000): 1464 rows over
                          product_name  original_price_inr
50        Samsung Slate 8GB RAM Silver           113586.17
97           Lenovo Pad 8GB RAM Silver            94787.00
246  Samsung Galaxy Tab 8GB RAM Silver           115081.13
360   Samsung Galaxy Tab 8GB RAM Black           110190.36
396         OnePlus iPad 4GB RAM Black            96871.02

Smartphones (cap ₹150,000): 7333 rows over
                     product_name  original_price_inr
3        Apple iPhone X 64GB Blue           201630.77
9     Apple iPhone XR 128GB Black           234658.90
15   Samsung Galaxy S7 16GB White       

In [293]:
print(f"NaN in original_price_inr: {df['original_price_inr'].isna().sum()}")

# Check the actual NaN rows
nan_rows = df[df["discounted_price_inr"].isna()]
print(nan_rows[["product_name", "original_price_inr", "discount_percent", "discounted_price_inr"]].head(10))

NaN in original_price_inr: 2982
                        product_name  original_price_inr  discount_percent  \
61     OnePlus OnePlus 6T 128GB Blue                 NaN             13.93   
190         Oppo R17 Pro 128GB Black                 NaN              7.87   
210   Xiaomi Redmi Note 4 16GB White                 NaN              0.00   
222     Samsung Galaxy S6 16GB Black                 NaN             20.19   
226    OnePlus OnePlus 5T 16GB Black                 NaN             33.86   
325          Oppo R17 Pro 64GB Black                 NaN              0.00   
327             Vivo Y95 128GB Black                 NaN             15.92   
404         Apple iPhone X 16GB Gold                 NaN             13.39   
412        Lenovo Pad 8GB RAM Silver                 NaN              0.00   
451  Motorola Moto G4 Plus 16GB Blue                 NaN              9.51   

     discounted_price_inr  
61                    NaN  
190                   NaN  
210                   NaN

In [294]:
# Check which subcategories these NaN prices belong to
print(df[df["original_price_inr"].isna()]["subcategory"].value_counts())

subcategory
Smartphones           2203
Laptops                247
Tablets                199
Smart Watch            176
Audio                  111
TV & Entertainment      46
Name: count, dtype: int64


In [295]:
# See the full distribution of Smartphones over cap
print(df[df["subcategory"] == "Smartphones"]["original_price_inr"]
      .describe().round(2))

count     70048.00
mean      53156.18
std       39410.24
min        1500.59
25%       26000.80
50%       40358.80
75%       83049.88
max      242371.11
Name: original_price_inr, dtype: float64


In [296]:
print(df[df["original_price_inr"].isna()].head(10).to_string())

        transaction_id order_date         customer_id   product_id                     product_name     category  subcategory     brand  original_price_inr  discount_percent  discounted_price_inr  quantity  subtotal_inr  final_amount_inr customer_city customer_state customer_tier customer_spending_tier customer_age_group payment_method  delivery_days delivery_type  is_prime_member  is_festival_sale      festival_name  customer_rating return_status  order_month  order_year  order_quarter  product_weight_kg  is_prime_eligible  product_rating
61   TXN_2018_00000062 2018-01-02  CUST_2018_00004931  PROD_000396    OnePlus OnePlus 6T 128GB Blue  Electronics  Smartphones   OnePlus                 NaN             13.93                   NaN         2           NaN               NaN        Mumbai    Maharashtra         Metro               Standard                NaN    Credit Card            2.0       Express             True             False                NaN              4.0     Delivered   

In [297]:
# Drop rows with NaN original_price_inr
df = df.dropna(subset=["original_price_inr"]).copy()  # ← added .copy()

# Recalculate
df["discounted_price_inr"] = (df["original_price_inr"] * (1 - df["discount_percent"] / 100)).round(2)
df["subtotal_inr"] = (df["discounted_price_inr"] * df["quantity"]).round(2)
df["final_amount_inr"] = df["subtotal_inr"]




Question 10
Payment methods contain inconsistent naming: 'UPI/PhonePe/GooglePay', 'Credit Card/CREDIT_CARD/CC', 
'Cash on Delivery/COD/C.O.D'. Standardize payment method categories and create a clean categorical hierarchy.


In [298]:
# ── Standardize payment methods ────────────────────
payment_standardize = {
    # UPI variants
    "UPI": "UPI", "PhonePe": "UPI", "GooglePay": "UPI", "Google Pay": "UPI",
    "UPI/PhonePe": "UPI", "UPI/GooglePay": "UPI",

    # Credit Card variants
    "Credit Card": "Credit Card", "CREDIT_CARD": "Credit Card", "CC": "Credit Card",

    # Debit Card variants
    "Debit Card": "Debit Card", "DEBIT_CARD": "Debit Card", "DC": "Debit Card",

    # COD variants
    "Cash on Delivery": "COD", "COD": "COD", "C.O.D": "COD",

    # Others
    "Wallet": "Wallet",
    "Net Banking": "Net Banking",
    "BNPL": "BNPL"
}

df["payment_method"] = df["payment_method"].map(payment_standardize).fillna(df["payment_method"])

# ── Create categorical hierarchy ───────────────────
payment_category = {
    "UPI":          "Digital Payment",
    "Wallet":       "Digital Payment",
    "Net Banking":  "Digital Payment",
    "Credit Card":  "Card Payment",
    "Debit Card":   "Card Payment",
    "BNPL":         "Pay Later",
    "COD":          "Cash on Delivery"
}

df["payment_category"] = df["payment_method"].map(payment_category).astype("category")

# ── Verify ─────────────────────────────────────────
print(df["payment_method"].value_counts())
print(f"\nNaN in payment_category: {df['payment_category'].isna().sum()}")

payment_method
COD            48160
Credit Card    16372
Debit Card     13642
UPI            12512
Net Banking     5827
Name: count, dtype: int64

NaN in payment_category: 0


Handling Nan - in customer age group

In [299]:
df["customer_age_group"] = df["customer_age_group"].fillna("Unknown")

Checking for object columns

In [300]:
df.select_dtypes(include="object").columns

Index(['transaction_id', 'customer_id', 'product_id', 'product_name',
       'category', 'subcategory', 'brand', 'customer_city', 'customer_state',
       'customer_tier', 'customer_spending_tier', 'customer_age_group',
       'payment_method', 'delivery_type', 'festival_name', 'return_status'],
      dtype='object')

In [301]:
# ── Optimize memory: convert to category dtype ───────
cat_columns = ["category", "subcategory", "customer_tier", 
               "customer_spending_tier", "customer_age_group",
               "payment_method", "delivery_type", 
               "festival_name", "return_status"]

df[cat_columns] = df[cat_columns].astype("category")

# Verify
print(df[cat_columns].dtypes)
print(f"\nMemory usage after optimization:")
print(df.memory_usage(deep=True).sum() / 1024**2, "MB")


category                  category
subcategory               category
customer_tier             category
customer_spending_tier    category
customer_age_group        category
payment_method            category
delivery_type             category
festival_name             category
return_status             category
dtype: object

Memory usage after optimization:
52.49823760986328 MB


In [302]:
# NaN summary for all columns
nan_summary = df.isna().sum()
nan_summary = nan_summary[nan_summary > 0].sort_values(ascending=False)

print(f"Total columns with NaN: {len(nan_summary)}")
print(f"Total rows in dataset: {df.shape[0]}")
print(f"\nNaN counts and percentages:")
print(pd.DataFrame({
    "NaN Count": nan_summary,
    "Percentage": (nan_summary / df.shape[0] * 100).round(2)
}))

Total columns with NaN: 2
Total rows in dataset: 96513

NaN counts and percentages:
                 NaN Count  Percentage
festival_name        65886       68.27
customer_rating      29332       30.39


In [303]:
df.to_csv("data_cleaning_2018.csv", index=False)
print("File saved successfully!")

File saved successfully!


In [304]:
df[df["category"] != "Electronics"][["order_year", "order_date", "category"]].value_counts().reset_index().sort_values("order_year")

,order_year,order_date,category,count
